# Step 4: download the satellite L3 SST (2018–2020)

This notebook fetches three products for 18–23°N, 86–93°E only: MetOp-B AVHRR (NASA PO.DAAC), MODIS Aqua and MODIS Terra (NOAA CoastWatch). Only the small box is read from each global file, so the downloads stay small.

You need: the newest `nbob_sst_study` zip, your NASA Earthdata username and password, and about 3 GB free in Google Drive.

Total time is roughly 2–5 hours. Each product has its own cell, and every cell can be stopped and run again later: finished files are skipped. Run the cells from top to bottom (Shift+Enter).

## 1. Clean start and install

In [ ]:
%cd /content
!rm -rf /content/nbob_sst_study /content/*.zip
!pip -q install xarray netCDF4 h5netcdf siphon earthaccess pandas

## 2. Upload the newest `nbob_sst_study` zip

In [ ]:
from google.colab import files
uploaded = files.upload()
zipname = list(uploaded.keys())[0]
!unzip -o -q "$zipname" -d /content
%cd /content/nbob_sst_study
!cat VERSION

**Check:** the cell above must print `2026-10-12a` or a later version.

## 3. Save to Google Drive
Allow access when asked.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/nbob_step4_l3'
os.makedirs(OUT, exist_ok=True)
print('Results will be saved in', OUT)

## 4. NASA Earthdata login (for MetOp-B)
Type your Earthdata username and password. They are written to a login file inside this temporary Colab machine only, which is how NASA's own tools log in. The file disappears when the session ends.

In [ ]:
import getpass, os, earthaccess
user = input('Earthdata username: ')
pw = getpass.getpass('Earthdata password: ')
netrc = os.path.expanduser('~/.netrc')
with open(netrc, 'w') as f:
    f.write(f'machine urs.earthdata.nasa.gov login {user} password {pw}\n')
os.chmod(netrc, 0o600)
auth = earthaccess.login(strategy='netrc')
print('Login OK' if auth.authenticated else 'Login FAILED: check username and password')

## 5. Quick check of the NOAA catalogue
This prints the first folders of the MODIS catalogue. It takes under a minute. If it shows a red error, send the last 20 lines and skip cells 7 and 8 for now; cell 6 does not depend on it.

In [ ]:
!python step4_l3_download.py --product aqua --inspect 2>&1 | tail -40

## 6. MetOp-B AVHRR (about 2,200 files)
The first run spends a few minutes listing the files. Then each line reports progress every 50 files.

In [ ]:
!python step4_l3_download.py --product metopb --out "$OUT" --workers 4 2>&1 | tee -a "$OUT/log_metopb.txt"

## 7. MODIS Aqua (about 2,200 files)
The first run reads the NOAA catalogue folder by folder, which can take 10–20 minutes before downloads start.

In [ ]:
!python step4_l3_download.py --product aqua --out "$OUT" --workers 4 2>&1 | tee -a "$OUT/log_aqua.txt"

## 8. MODIS Terra (about 2,200 files)

In [ ]:
!python step4_l3_download.py --product terra --out "$OUT" --workers 4 2>&1 | tee -a "$OUT/log_terra.txt"

## 9. Inventory
Counts the files and the share of valid pixels per product and phase.

In [ ]:
!python step4_l3_download.py --inventory --out "$OUT" 2>&1 | tee "$OUT/log_inventory.txt"

## 10. Download the small summary files
Send `step4_results.zip` back in the chat. The large data files stay in your Drive for the next step.

In [ ]:
!cd "$OUT" && zip -q /content/step4_results.zip inventory_*.csv product_attributes.txt granules_*.csv failed_*.csv log_*.txt
files.download('/content/step4_results.zip')

## If something fails
- A cell stopped or Colab disconnected: run cells 1–4 again, then the same download cell. Finished files are skipped.
- `FAILED` lines for a few files: run the same cell again; it retries only the missing files.
- A red error: copy the last 20 lines and send them in the chat.